# Random Forest from Scratch

The aim of this notebook is to implement **random forest classifier and regressor** algorithms from scratch using only NumPy. 

In [1]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

## 1. Random Forest Classifier

In [2]:
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import accuracy_score
from decision_tree import DecisionTreeClassifier

In [3]:
df_classification = pd.DataFrame(load_breast_cancer().data, columns=load_breast_cancer().feature_names)
df_classification['target'] = load_breast_cancer().target # binary classification: malignant or benign

In [4]:
df_classification.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0


In [5]:
# Prepare the data: target variable and features
X_classification = df_classification.drop("target", axis=1).values # features
y_classification = df_classification["target"].values # target

In [6]:
# Split datasets into training and testing sets
Xc_train, Xc_test, yc_train, yc_test = train_test_split(X_classification, y_classification, test_size=0.2, random_state=42)

In [7]:
# Boostrap sampling function
def boostrap_sampling(X, y):
    # Number of training exemples
    n_examples = len(X[:, 0])

    # Sample indices with replacement
    sampled_indices = np.random.choice(n_examples, n_examples, replace=True)

    # Get the sampled data
    X_sample = X[sampled_indices]
    y_sample = y[sampled_indices]

    # Get the out-of-bag (OOB) indices
    oob_indices = np.setdiff1d(np.arange(n_examples), sampled_indices) # return the indices that are not in the sampled_indices

    return X_sample, y_sample, oob_indices

In [8]:
# Test with the training set
X_sample, y_sample, oob_indices = boostrap_sampling(Xc_train, yc_train)
print("Sampled data shape:", X_sample.shape)
print("Sampled target shape:", y_sample.shape)
print("Out-of-bag indices:", oob_indices)
print("Out-of-bag shape:", oob_indices.shape)

Sampled data shape: (455, 30)
Sampled target shape: (455,)
Out-of-bag indices: [  2   3  10  13  16  20  23  26  33  34  35  38  40  44  45  47  48  50
  54  57  61  62  64  67  68  73  74  82  83  85  86  91  96  97  98  99
 102 104 113 116 117 118 119 122 124 125 131 132 135 139 142 143 145 146
 147 151 152 156 157 164 165 169 171 178 179 182 192 197 199 204 210 211
 220 222 223 224 227 232 234 235 236 239 242 246 251 252 255 256 262 263
 267 270 275 280 285 292 296 305 311 312 313 319 320 321 322 324 325 326
 331 332 338 340 342 345 346 347 348 350 353 354 355 358 361 363 368 371
 372 375 378 379 381 385 389 391 397 398 399 400 402 403 405 406 408 409
 411 413 416 418 423 425 429 432 435 438 443 444 449 450 451 452 453 454]
Out-of-bag shape: (162,)


Out-of-bag shape / Sampled data shape = 168 / 455 ≈ 36.9%, exactly what we expected. 

In [9]:
assert len(X_sample) == len(Xc_train) # len(unique sampled indices) + len(oob_indices) == n

In [10]:
# Training loop
def fit_classifier(X, y, n_estimators=10, max_features=5, max_depth=3, min_samples_split=2, min_samples_leaf=1):
    # List to store trees
    forest = []

    # Loop over each tree
    for _ in range(n_estimators):
        X_sample, y_sample, oob_indices = boostrap_sampling(X, y)
        model = DecisionTreeClassifier(max_depth=max_depth, max_features=max_features, min_samples_split=min_samples_split, min_samples_leaf=min_samples_leaf)
        model.fit(X_sample, y_sample)
        forest.append((model, oob_indices))

    return forest

In [11]:
fit_classifier(Xc_train, yc_train)

[(<decision_tree.DecisionTreeClassifier at 0x136f02cf0>,
  array([  2,   5,   6,  12,  13,  19,  20,  23,  25,  28,  29,  30,  32,
          38,  39,  42,  44,  48,  49,  51,  52,  53,  58,  60,  63,  65,
          66,  68,  69,  72,  73,  76,  77,  79,  81,  82,  86,  89,  91,
         102, 105, 106, 107, 108, 110, 111, 113, 114, 115, 116, 119, 120,
         128, 130, 131, 135, 138, 141, 143, 146, 149, 152, 159, 160, 161,
         162, 163, 164, 176, 181, 182, 183, 184, 190, 194, 195, 198, 200,
         201, 203, 204, 205, 209, 210, 211, 212, 213, 215, 217, 222, 223,
         225, 226, 227, 232, 235, 236, 240, 242, 250, 254, 255, 256, 257,
         260, 264, 265, 267, 269, 271, 274, 276, 277, 279, 281, 282, 283,
         286, 288, 291, 295, 298, 299, 300, 304, 306, 309, 311, 313, 314,
         317, 322, 324, 325, 326, 330, 332, 333, 336, 337, 339, 341, 343,
         344, 345, 346, 356, 361, 366, 368, 369, 371, 373, 374, 376, 377,
         381, 382, 383, 384, 386, 387, 394, 395, 398, 3

In [12]:
# Prediction by majority vote
def predict_classifier(forest, X):
    # List to store predictions from each tree
    tree_predictions = []

    # Loop over each tree in the forest
    for model, _ in forest:
        tree_predictions.append(model.predict(X))

    # Stack predictions from all trees into a 2D array (matrix)
    tree_predictions = np.stack(tree_predictions, axis=1) # shape (n_samples, n_estimators)
    tree_predictions = tree_predictions.astype(int)

    # Majority vote: for each sample, find the most common prediction among the trees
    predictions = np.apply_along_axis(lambda x: np.bincount(x).argmax(), axis=1, arr=tree_predictions)

    return predictions

In [13]:
y_pred = predict_classifier(fit_classifier(Xc_train, yc_train), Xc_test)
accuracy = accuracy_score(yc_test, y_pred)
print("Accuracy:", accuracy)

Accuracy: 0.956140350877193


In [14]:
# Compute the OOB error
def oob_score(forest, X, y):
    # Initialize an empty list to store OOB predictions for each sample
    n_samples = len(X)
    oob_predictions = [[] for _ in range(n_samples)]

    # For each tree in the forest, get the OOB indices and make predictions for those samples
    for model, oob_indices in forest:
        predictions = model.predict(X[oob_indices]).astype(int)
        for idx, pred in zip(oob_indices, predictions):
            oob_predictions[idx].append(pred)

    # For each sample, find the most common prediction among the trees that did not use it for training
    final_oob_predictions = np.zeros(n_samples, dtype=int)
    for i in range(n_samples):
        # Get the predictions from trees that did not use this sample for training
        tree_preds = np.array(oob_predictions[i])
        if len(tree_preds) > 0: # if there are any OOB predictions for this sample
            final_oob_predictions[i] = np.bincount(tree_preds).argmax() # majority vote

    # Calculate OOB accuracy
    oob_accuracy = accuracy_score(y, final_oob_predictions)
    return oob_accuracy

In [15]:
oob_accuracy = oob_score(fit_classifier(Xc_train, yc_train), Xc_train, yc_train)
print("OOB Accuracy:", oob_accuracy)

OOB Accuracy: 0.9120879120879121


In [16]:
def feature_importance(forest, n_features):
    # Initialize an array to store feature importance scores
    importance_scores = np.zeros(n_features)

    # Recursive function to traverse the tree and update importance scores
    def _traverse(node):
        if node.value is not None:  # leaf, we stop traversing
            return
        importance_scores[node.feature] += 1
        _traverse(node.left)
        _traverse(node.right)

    # For each tree in the forest, check if the node is a leaf node or not, if not, get the feature index used for splitting and update the importance score for that feature
    for model, _ in forest:
        # Check if the node is a leaf node
        _traverse(model.tree)

    # Normalize importance scores
    importance_scores /= np.sum(importance_scores)

    return importance_scores

In [18]:
features = load_breast_cancer().feature_names
importances = feature_importance(fit_classifier(Xc_train, yc_train), Xc_train.shape[1])

sorted_idx = np.argsort(importances)[::-1]
for i in sorted_idx:
    print(f"{features[i]}: {importances[i]:.4f}")

worst concave points: 0.1212
worst concavity: 0.0909
worst area: 0.0758
worst perimeter: 0.0758
mean concavity: 0.0606
worst smoothness: 0.0606
mean area: 0.0606
mean texture: 0.0455
area error: 0.0455
mean radius: 0.0455
worst compactness: 0.0455
fractal dimension error: 0.0303
worst radius: 0.0303
worst texture: 0.0303
worst symmetry: 0.0303
perimeter error: 0.0303
radius error: 0.0303
mean perimeter: 0.0303
concavity error: 0.0152
texture error: 0.0152
mean symmetry: 0.0152
mean concave points: 0.0152
mean compactness: 0.0000
mean smoothness: 0.0000
worst fractal dimension: 0.0000
mean fractal dimension: 0.0000
compactness error: 0.0000
concave points error: 0.0000
symmetry error: 0.0000
smoothness error: 0.0000


## 2. Random Forest Regressor